# 02 — ML Frame Scoring (Embedding)

Requires: `pip install -r requirements-ml.txt` (then install PyTorch for your hardware — see that file).

FrameAxis-inspired bipolar embedding scoring using LaBSE sentence-transformer cosine similarity.
Complements the dictionary-based frame flags with continuous semantic scores.

Results are saved to `data/processed/` and used as supplementary figures in `03_paper_figures.ipynb`.

| Section | What it does | Output |
|---------|-------------|--------|
| 1 | Load parquet (text + metadata only) + stratified sample | — |
| 2 | Bipolar embedding scoring (LaBSE) | `ml_frame_scores_embedding.parquet` |
| 3 | Dictionary validation (t-tests, Cohen's κ) | printed table |
| 4 | Combined flags (keyword ∩ embedding > 0) | `ml_frame_confirmed.parquet` |

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dictionaries import FRAME_COLS, FRAME_DICTS

INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"

PREPROCESSED           = INTERIM   / "gdelt_preprocessed.parquet"
TEMP_EMBEDDING_PATH    = INTERIM   / "full_embedding_data.parquet"
ML_EMBEDDING_PATH      = PROCESSED / "ml_frame_scores_embedding.parquet"
ML_EMBEDDING_FULL_PATH = PROCESSED / "ml_frame_scores_embedding_full.parquet"
ML_CONFIRMED_PATH      = PROCESSED / "ml_frame_confirmed.parquet"

FRAME_NAMES = [c.replace("frame_", "") for c in FRAME_COLS]

NEEDED_COLS = ["DocumentIdentifier", "headline_text", "Quotations",
               "dominant_frame", "month", "region"] + FRAME_COLS

SAMPLE_N    = None   # set to None for full corpus (~8–15 min on GPU)
RANDOM_SEED = 42

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("No GPU detected — running on CPU (will be slow)")
except ImportError:
    DEVICE = "cpu"
    print("torch not installed — run: pip install -r requirements-ml.txt")

GPU: AMD Radeon RX 9070 XT


## Section 1 — Load text (memory-efficient)

Loads only `Quotations` + frame metadata columns from the preprocessed parquet.
This is ~20× smaller than loading the full 2 GB parquet.

In [2]:
df_meta = pd.read_parquet(PREPROCESSED, columns=NEEDED_COLS)
mem_mb = df_meta.memory_usage(deep=True).sum() / 1e6
print(f"Loaded {len(df_meta):,} articles  ({mem_mb:.0f} MB in memory)")

if SAMPLE_N is not None and SAMPLE_N < len(df_meta):
    n_months = df_meta["month"].nunique()
    sample_per_month = max(1, SAMPLE_N // n_months)
    ym_str = df_meta["month"].astype(str)
    df_sample = pd.concat(
        [g.sample(min(len(g), sample_per_month), random_state=RANDOM_SEED)
         for _, g in df_meta.groupby(ym_str, observed=True)],
        ignore_index=True,
    )
    if len(df_sample) > SAMPLE_N:
        df_sample = df_sample.sample(SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    df_sample = df_meta.reset_index(drop=True)

texts = (
    df_sample["headline_text"].fillna("") + " " + df_sample["Quotations"].fillna("")
).str.strip().tolist()

print(f"Sample: {len(df_sample):,} articles across {df_sample['month'].nunique()} months")

Loaded 1,116,091 articles  (419 MB in memory)
Sample: 1,116,091 articles across 44 months


## Section 2 — Embedding scoring

FrameAxis-inspired cosine similarity between article texts and frame keyword centroids.
Scores are in [−1, 1]; higher = semantically closer to the frame.

Skipped if `ml_frame_scores_embedding.parquet` already exists.

In [5]:
from src.framing_scores import assign_frame_scores_embedding
from src.dictionaries import FRAME_DICTS_NEG

if TEMP_EMBEDDING_PATH.exists():
    emb_scores = pd.read_parquet(TEMP_EMBEDDING_PATH)
    print(f"Loaded from interim: {TEMP_EMBEDDING_PATH.name}  ({len(emb_scores):,} rows)")
else:
    print(f"Scoring {len(texts):,} articles (bipolar FrameAxis, GPU batches internally)...")
    emb_scores = assign_frame_scores_embedding(texts, FRAME_DICTS, frame_dicts_neg=FRAME_DICTS_NEG)
    emb_scores.to_parquet(TEMP_EMBEDDING_PATH, index=False)
    print(f"Saved {len(emb_scores):,} rows → {TEMP_EMBEDDING_PATH}")

emb_scores

Loaded from interim: full_embedding_data.parquet  (1,116,091 rows)


,innovation_opportunity,risk_safety,regulation_governance,rights_privacy,economic_competition_labour,misinformation_integrity
0,0.002621,0.022771,0.006501,0.051096,-0.078427,0.060937
1,0.005761,0.019260,0.052804,0.048389,0.018018,0.048808
2,-0.048220,0.103527,0.046885,0.069374,0.050480,0.021588
3,-0.005344,-0.012254,-0.021658,0.025778,0.017607,0.029921
4,-0.037350,0.058464,0.062117,0.052672,-0.015786,0.096100
...,...,...,...,...,...,...
1116086,0.040208,0.051235,-0.035752,0.031198,0.030770,0.029166
1116087,0.055196,0.012847,-0.026724,0.029027,0.010473,0.063677
1116088,-0.052293,0.065101,0.038384,0.120362,0.070923,0.045723
1116089,-0.142406,0.070051,0.026996,0.064311,0.037793,0.133631


In [4]:
out_path = ML_EMBEDDING_FULL_PATH if SAMPLE_N is None else ML_EMBEDDING_PATH

if out_path.exists():
    scores_emb = pd.read_parquet(out_path)
    print(f"Loaded from cache: {out_path.name}  ({len(scores_emb):,} rows)")
else:
    meta_cols = ["DocumentIdentifier", "dominant_frame", "month", "region"] + FRAME_COLS
    scores_emb = pd.concat(
        [df_sample[meta_cols].reset_index(drop=True), emb_scores.reset_index(drop=True)],
        axis=1,
    )
    scores_emb.to_parquet(out_path, index=False)
    print(f"Saved {len(scores_emb):,} rows → {out_path}")

Saved 1,116,091 rows → /home/Brewen/cwd/tud/GenAI-GDELT/data/processed/ml_frame_scores_embedding_full.parquet


## Section 3 — Dictionary validation

For each frame: mean bipolar embedding score for articles where the keyword flag fires vs.
where it does not. Welch's t-test.

**Expected:** embedding scores should be significantly higher when the keyword flag fires,
validating the dictionary-based approach.

Also reports Cohen's κ for dominant frame agreement between keyword and embedding methods.

In [6]:
from scipy import stats
from sklearn.metrics import cohen_kappa_score

# dominant_frame_emb: argmax of embedding scores per article
scores_emb["dominant_frame_emb"] = scores_emb[FRAME_NAMES].idxmax(axis=1)

print("Dictionary validation: mean bipolar embedding score for keyword-matched vs. unmatched articles\n")
cols = f"{'Frame':<38} {'n_hit':>6}  {'emb_hit':>8} {'emb_no':>8} {'emb_p':>10}"
print(cols)
print("-" * len(cols))

for name in FRAME_NAMES:
    col_kw = f"frame_{name}"
    if col_kw not in scores_emb.columns:
        continue
    has_flag = scores_emb[col_kw] > 0
    n_hit, n_no = has_flag.sum(), (~has_flag).sum()
    if n_hit < 2 or n_no < 2:
        print(f"{name:<38}  (insufficient data: {n_hit} matches)")
        continue

    emb_hit = scores_emb.loc[has_flag,  name].mean()
    emb_no  = scores_emb.loc[~has_flag, name].mean()
    _, p_emb = stats.ttest_ind(
        scores_emb.loc[has_flag,  name],
        scores_emb.loc[~has_flag, name],
        equal_var=False,
    )
    print(f"{name:<38} {n_hit:>6}  {emb_hit:8.3f} {emb_no:8.3f} {p_emb:10.2e}")

print()
valid = scores_emb["dominant_frame"].notna()
kw_arr  = scores_emb.loc[valid, "dominant_frame"].values
emb_arr = scores_emb.loc[valid, "dominant_frame_emb"].values

print(f"Cohen's κ  keyword vs. embedding  (n={valid.sum():,} articles with ≥1 keyword hit)")
try:
    print(f"  κ = {cohen_kappa_score(kw_arr, emb_arr):.3f}")
except Exception as e:
    print(f"  Failed: {e}")

Dictionary validation: mean bipolar embedding score for keyword-matched vs. unmatched articles

Frame                                   n_hit   emb_hit   emb_no      emb_p
---------------------------------------------------------------------------
innovation_opportunity                 141203     0.032    0.021   0.00e+00
risk_safety                            160641     0.044    0.022   0.00e+00
regulation_governance                  316440     0.012   -0.009   0.00e+00
rights_privacy                          44958     0.065    0.048   0.00e+00
economic_competition_labour            131261     0.018    0.012   0.00e+00
misinformation_integrity                45795     0.063    0.008   0.00e+00

Cohen's κ  keyword vs. embedding  (n=564,922 articles with ≥1 keyword hit)
  κ = 0.129


## Section 4 — Combined flags (keyword ∩ embedding)

High-precision frame assignment: an article is confirmed in frame X only when the keyword
dictionary matches at least one term from frame X AND the bipolar embedding score for frame X
is positive (i.e., the article text is semantically closer to the positive semantic pole than
to the negative pole).

Methodological rationale: keywords provide recall; embeddings provide semantic confirmation of
keyword hits, filtering out incidental matches. The threshold of 0 follows directly from the
bipolar scoring — no tuning required.

Saved to `ml_frame_confirmed.parquet` for use in `03_paper_figures.ipynb` (Fig S5).
Requires full-corpus embedding run (`SAMPLE_N = None` in Section 2).

In [9]:
if not ML_EMBEDDING_FULL_PATH.exists():
    print("Skipping — run with SAMPLE_N=None first to generate ml_frame_scores_embedding_full.parquet.")
else:
    kw_cols = ["DocumentIdentifier"] + FRAME_COLS
    kw_full = pd.read_parquet(PREPROCESSED, columns=kw_cols)
    emb_full = pd.read_parquet(ML_EMBEDDING_FULL_PATH)

    merged = kw_full.merge(
        emb_full[["DocumentIdentifier", "month", "region"] + FRAME_NAMES],
        on="DocumentIdentifier", how="inner"
    )

    for name in FRAME_NAMES:
        merged[f"frame_{name}"] = (
            (merged[f"frame_{name}"] > 0) & (merged[name] > 0)
        ).astype(int)

    sizes = pd.Series({f"frame_{n}": len(v) for n, v in FRAME_DICTS.items()})
    norm = merged[FRAME_COLS].div(sizes, axis=1)

    has_positive = norm.gt(0).any(axis=1)
    dominant_idx = norm[has_positive].idxmax(axis=1).str.replace("frame_", "", regex=False)
    merged["dominant_frame_confirmed"] = dominant_idx.reindex(merged.index)

    out = merged[["DocumentIdentifier", "month", "region",
                  "dominant_frame_confirmed"] + FRAME_COLS]
    out.to_parquet(ML_CONFIRMED_PATH, index=False)

    n_conf = (out[FRAME_COLS] > 0).any(axis=1).sum()
    print(f"Saved {len(out):,} rows → {ML_CONFIRMED_PATH.name}")
    print(f"  Articles with ≥1 confirmed frame: {n_conf:,} ({n_conf/len(out):.1%})")
    print(f"\n  Confirmed frame breakdown:")
    for name in FRAME_NAMES:
        n = (out[f"frame_{name}"] > 0).sum()
        print(f"    {name:<38} {n:>6,}")

Saved 1,116,091 rows → ml_frame_confirmed.parquet
  Articles with ≥1 confirmed frame: 455,349 (40.8%)

  Confirmed frame breakdown:
    innovation_opportunity                 106,708
    risk_safety                            136,379
    regulation_governance                  185,245
    rights_privacy                         40,055
    economic_competition_labour            85,657
    misinformation_integrity               39,236
